<a href="https://colab.research.google.com/github/cube2ishaank-debug/FoodScopeAI/blob/main/USE_FoodScopeAI_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# ==========================================
# 1. INITIALIZE THE ACTUAL RESNET-18 MODEL
# ==========================================
# Load the official ResNet-18 architecture backbone
model = models.resnet18()

# Get the number of input features going into the final layer (will be 512)
num_features = model.fc.in_features

# FIXED: Change the final layer to output 27 classes instead of the default 1000
model.fc = nn.Linear(num_features, 27)

# ==========================================
# 2. LOAD THE MODEL WEIGHTS
# ==========================================
# Load the file dictionary securely
checkpoint = torch.load(
    'fridge_model_v1.pth',
    map_location=torch.device('cpu'),
    weights_only=True
)

# Extract and apply the weights from the nested key
model.load_state_dict(checkpoint["model_state_dict"])

# Extract metadata class names saved in the file
class_names = checkpoint.get("class_names", [])
print(f"Successfully loaded ResNet-18 model!")
print(f"Total classes found in weights: {len(class_names)}")

# Set the model to evaluation/inference mode
model.eval()

# ==========================================
# 3. PREPARE THE INPUT IMAGE
# ==========================================
# Standard ResNet preprocessing pipeline
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def predict_image(image_path):
    # Open and convert image to RGB format
    image = Image.open(image_path).convert('RGB')

    # Preprocess and add a batch dimension: [3, 224, 224] -> [1, 3, 224, 224]
    input_tensor = image_transforms(image).unsqueeze(0)

    # ==========================================
    # 4. RUN INFERENCE & MAP PREDICTION
    # ==========================================
    with torch.no_grad():
        # Get raw logit outputs from the model
        outputs = model(input_tensor)

        # Convert raw outputs to index probability
        _, predicted_idx = torch.max(outputs, 1)
        class_index = predicted_idx.item()

    # Print the prediction using the extracted class names
    if class_names and class_index < len(class_names):
        predicted_label = class_names[class_index]
        print(f"\nPrediction for '{image_path}':")
        print(f"👉 {predicted_label} (Class Index: {class_index})")
    else:
        print(f"\nPrediction for '{image_path}': Class Index {class_index}")

# ==========================================
# 5. EXECUTE PREDICTION
# ==========================================
# Uncomment this line and add your photo path to test it!
# predict_image("test_fridge_item.jpg")


Successfully loaded ResNet-18 model!
Total classes found in weights: 27


In [ ]:
predict_image("/content/image_name.png")


Prediction for '/content/Raw-Meat-Yummies-01-Rare-Steak.jpg':
👉 raw_fish (Class Index: 21)
